In [2]:
"""
================================================================================================
NOTEBOOK 03: Ensemble Construction & Statistical Significance Testing
================================================================================================
Author: Mary Wangoi Mwangi (122174)
================================================================================================
Goal:
    - Build a stacking ensemble combining the three trained models from Notebook 02
    - Perform statistical significance testing to validate model differences.

Key Actions:
    1. Load trained models and data splits saved from Notebook 02
    2. Build stacking ensemble — RF + XGBoost + LR as base learners
    3. Optimise ensemble threshold using validation set
    4. Perform statistical significance testing (Friedman + Wilcoxon)
    5. Save the ensemble model for final evaluation in Notebook 04

Outcome:
    Trained stacking ensemble saved to models/ folder.
    Statistical test results confirming model differences.
    Ready for final test set evaluation in Notebook 04.
================================================================================================
"""

# ============================================================
# SECTION 1: LIBRARY IMPORTS & GLOBAL CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (recall_score, precision_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, average_precision_score,
                             precision_recall_curve)
from scipy.stats import wilcoxon, friedmanchisquare
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')

# Reproducibility seed
SEED = 42

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Libraries loaded successfully.")
print(f"   pandas      : {pd.__version__}")
print(f"   numpy       : {np.__version__}")
print(f"   sklearn     : {__import__('sklearn').__version__}")
print(f"   scipy       : {__import__('scipy').__version__}")

Libraries loaded successfully.
   pandas      : 2.3.3
   numpy       : 2.4.3
   sklearn     : 1.8.0
   scipy       : 1.17.1


In [3]:
# ============================================================
# SECTION 2: LOAD MODELS, METADATA & DATA SPLITS

# Load trained models, metadata and data splits directly from disk saved from Notebook 02
# No rebuilding required — splits are identical to Notebook 02
# ============================================================

# ---- Load trained models ----
best_lr  = joblib.load('../models/best_lr.pkl')
best_rf  = joblib.load('../models/best_rf.pkl')
best_xgb = joblib.load('../models/best_xgb.pkl')

print("Models loaded successfully.")
print(f"   best_lr.pkl  : {type(best_lr).__name__}")
print(f"   best_rf.pkl  : {type(best_rf).__name__}")
print(f"   best_xgb.pkl : {type(best_xgb).__name__}")


# ---- Load metadata ----
with open('../models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

optimal_thresholds = metadata['optimal_thresholds']
cat_cols           = metadata['feature_info']['cat_cols']
num_cols           = metadata['feature_info']['num_cols']

print(f"\nMetadata loaded successfully.")
print(f"   Optimal thresholds:")
for name, thresh in optimal_thresholds.items():
    print(f"      {name:<30} : {thresh}")


# ---- Load data splits directly from disk ----
X_train = pd.read_csv('../data/splits/X_train.csv')
X_val   = pd.read_csv('../data/splits/X_val.csv')
X_test  = pd.read_csv('../data/splits/X_test.csv')

y_train = pd.read_csv('../data/splits/y_train.csv').squeeze()
y_val   = pd.read_csv('../data/splits/y_val.csv').squeeze()
y_test  = pd.read_csv('../data/splits/y_test.csv').squeeze()

print(f"\nData splits loaded successfully:")
print(f"   Training   (70%) : {X_train.shape[0]:,} rows, "
      f"{X_train.shape[1]} features")
print(f"   Validation (15%) : {X_val.shape[0]:,} rows, "
      f"{X_val.shape[1]} features")
print(f"   Test       (15%) : {X_test.shape[0]:,} rows, "
      f"{X_test.shape[1]} features")

print(f"\nClass distribution confirmed:")
print(f"   Train High-Severity : {y_train.sum():,} "
      f"({y_train.mean()*100:.1f}%)")
print(f"   Val   High-Severity : {y_val.sum():,}   "
      f"({y_val.mean()*100:.1f}%)")
print(f"   Test  High-Severity : {y_test.sum():,}   "
      f"({y_test.mean()*100:.1f}%)")

print(f"\nAll data loaded. Ready for ensemble construction.")

Models loaded successfully.
   best_lr.pkl  : Pipeline
   best_rf.pkl  : Pipeline
   best_xgb.pkl : Pipeline

Metadata loaded successfully.
   Optimal thresholds:
      Logistic Regression            : 0.55
      Balanced Random Forest         : 0.51
      XGBoost                        : 0.48

Data splits loaded successfully:
   Training   (70%) : 8,621 rows, 28 features
   Validation (15%) : 1,847 rows, 28 features
   Test       (15%) : 1,848 rows, 28 features

Class distribution confirmed:
   Train High-Severity : 1,331 (15.4%)
   Val   High-Severity : 285   (15.4%)
   Test  High-Severity : 285   (15.4%)

All data loaded. Ready for ensemble construction.


In [4]:
# ============================================================
# SECTION 3: STACKING ENSEMBLE CONSTRUCTION

# Build stacking ensemble with RF, XGBoost and LR as base learners and LR as meta-learner

# Architecture:
#   Level 0 (Base Learners):
#     - Balanced Random Forest — best ROC AUC and Recall
#     - XGBoost               — best Precision
#     - Logistic Regression   — linear boundary complement
#   Level 1 (Meta-Learner):
#     - Logistic Regression   — learns optimal combination of base learner predictions

# Design decisions:
#   - stack_method='predict_proba' — passes probabilities to meta-learner for richer signal than binary predictions
#   - passthrough=False — meta-learner only sees base model predictions, not original features — prevents overfitting
#   - cv=5 — 5-fold CV used to generate meta-features ensuring no data leakage into meta-learner
# ============================================================

from sklearn.ensemble import StackingClassifier

print("STACKING ENSEMBLE CONSTRUCTION")
print("="*55)

# ---- Define base learners ----
# Using already tuned pipelines from Notebook 02
base_learners = [
    ('rf',  best_rf),
    ('xgb', best_xgb),
    ('lr',  best_lr)
]

# ---- Define meta-learner ----
# Logistic Regression with class_weight balanced
# C=0.5 — moderate regularisation prevents meta-overfitting
meta_learner = LogisticRegression(
    random_state=SEED,
    max_iter=1000,
    class_weight='balanced',
    C=0.5
)

# ---- Build stacking classifier ----
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)

# ---- Fit on training data ----
print("\n Fitting stacking ensemble on training data...")
print("This may take several minutes...")
stacking_clf.fit(X_train, y_train)

print("\n Stacking ensemble fitted successfully.")
print(f"\n Ensemble architecture:")
print(f"   Base learners  : RF, XGBoost, LR")
print(f"   Meta-learner   : Logistic Regression")
print(f"   CV folds       : 5")
print(f"   Stack method   : predict_proba")
print(f"   Passthrough    : False")

STACKING ENSEMBLE CONSTRUCTION

 Fitting stacking ensemble on training data...
This may take several minutes...

 Stacking ensemble fitted successfully.

 Ensemble architecture:
   Base learners  : RF, XGBoost, LR
   Meta-learner   : Logistic Regression
   CV folds       : 5
   Stack method   : predict_proba
   Passthrough    : False


In [5]:
# ============================================================
# SECTION 4: ENSEMBLE THRESHOLD OPTIMISATION

# Evaluate ensemble on validation set at default threshold
# Apply granular threshold search — same methodology as Notebook 02 — Precision >= 0.25, Maximise Recall
# ============================================================

print("ENSEMBLE VALIDATION EVALUATION")
print("="*55)

# ---- Get ensemble probabilities on validation set ----
ensemble_proba_val = stacking_clf.predict_proba(X_val)[:, 1]


# ---- Default threshold 0.50 ----
ensemble_pred_default = stacking_clf.predict(X_val)

print("\nEnsemble at Default Threshold (0.50):")
print(f"   Recall    : {recall_score(y_val, ensemble_pred_default):.3f}")
print(f"   Precision : {precision_score(y_val, ensemble_pred_default):.3f}")
print(f"   F1 Score  : {f1_score(y_val, ensemble_pred_default):.3f}")
print(f"   ROC AUC   : {roc_auc_score(y_val, ensemble_proba_val):.3f}")


# ---- Granular threshold search ----
print(f"\n{'='*55}")
print("GRANULAR THRESHOLD SEARCH (0.35 to 0.65, step=0.01)")
print("Constraint : Precision >= 0.25")
print("Objective  : Maximise Recall subject to constraint")
print("="*55)

thresholds_fine    = np.arange(0.35, 0.66, 0.01)
best_recall        = 0
best_threshold     = 0.50
best_precision     = 0
best_f1            = 0

print(f"\n{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'F1':<10}")
print(f"{'-'*44}")

for thresh in thresholds_fine:
    y_pred = (ensemble_proba_val >= thresh).astype(int)
    if y_pred.sum() == 0:
        continue

    rec  = recall_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred)

    print(f"{thresh:<12.2f} {rec:<10.3f} {prec:<12.3f} {f1:<10.3f}")

    if prec >= 0.25 and rec > best_recall:
        best_recall    = rec
        best_threshold = thresh
        best_precision = prec
        best_f1        = f1

print(f"\n{'='*55}")
print(f"OPTIMAL ENSEMBLE THRESHOLD")
print(f"{'='*55}")
print(f"   Threshold : {best_threshold:.2f}")
print(f"   Recall    : {best_recall:.3f}")
print(f"   Precision : {best_precision:.3f}")
print(f"   F1 Score  : {best_f1:.3f}")
print(f"   ROC AUC   : {roc_auc_score(y_val, ensemble_proba_val):.3f}")

# Store optimal threshold
optimal_thresholds['Stacking Ensemble'] = best_threshold

ENSEMBLE VALIDATION EVALUATION

Ensemble at Default Threshold (0.50):
   Recall    : 0.621
   Precision : 0.249
   F1 Score  : 0.355
   ROC AUC   : 0.710

GRANULAR THRESHOLD SEARCH (0.35 to 0.65, step=0.01)
Constraint : Precision >= 0.25
Objective  : Maximise Recall subject to constraint

Threshold    Recall     Precision    F1        
--------------------------------------------
0.35         0.926      0.182        0.304     
0.36         0.916      0.184        0.306     
0.37         0.898      0.186        0.309     
0.38         0.884      0.191        0.314     
0.39         0.856      0.193        0.315     
0.40         0.832      0.196        0.318     
0.41         0.814      0.202        0.324     
0.42         0.804      0.209        0.331     
0.43         0.772      0.212        0.332     
0.44         0.751      0.214        0.334     
0.45         0.730      0.218        0.335     
0.46         0.709      0.222        0.338     
0.47         0.702      0.231        0.34